# Chapter 7 — Transfer Learning for EO

## Learning Objectives
- Understand why ImageNet pretrained models transfer to satellite imagery
- Implement 3 strategies: linear probe, full fine-tune, progressive fine-tune
- Adapt input channels for multispectral data (3→10 bands)
- Compare scratch vs pretrained on EuroSAT
- Understand domain shift and when transfer learning fails

## Estimated Duration: Theory 2h | Practical 3h | Total 5h
## Difficulty: Intermediate

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision torchgeo matplotlib seaborn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from torchgeo.datasets import EuroSAT
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FAST_MODE = DEVICE == 'cpu'
DATA_ROOT = Path('./data') if not IN_COLAB else Path('/content/data')
print(f'Device: {DEVICE} | Fast mode: {FAST_MODE}')

## 7.1 — Why Does ImageNet Transfer to EO?

ImageNet contains 1.28M images of objects, animals, scenes.
Sentinel-2 land cover images are completely different — or are they?

What transfers from ImageNet:
  Early layers: edge detectors, texture filters (universal low-level features)
  Middle layers: shape parts, material textures (partially transferable)
  Late layers: object parts, semantic concepts (less relevant to land cover)

Key experiments (Marmanis et al., 2015; Yosinski et al., 2014):
  - Frozen ImageNet features + linear classifier on EO data: surprisingly good
  - Fine-tuning dramatically improves over random init
  - The more EO data, the less the pretrained features matter
  - For small EO datasets (<10k images), always use pretrained features

In [ ]:
# Strategy comparison: scratch vs linear probe vs full finetune
EUROSAT_ROOT = DATA_ROOT / 'eurosat'
EUROSAT_ROOT.mkdir(parents=True, exist_ok=True)

train_ds = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)
val_ds   = EuroSAT(root=EUROSAT_ROOT, split='val',   download=True)
NUM_CLASSES = len(train_ds.classes)

def collate(batch):
    imgs = torch.stack([b['image'][:3].float() / 10000.0 for b in batch])
    lbls = torch.tensor([b['label'] for b in batch])
    return imgs, lbls

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, collate_fn=collate, num_workers=0)

def build_resnet18(mode='finetune'):
    from torchvision.models import resnet18, ResNet18_Weights
    if mode == 'scratch':
        model = resnet18(weights=None)
    else:
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(512, NUM_CLASSES)
    if mode == 'linear_probe':
        for name, p in model.named_parameters():
            if 'fc' not in name:
                p.requires_grad_(False)
    return model

def quick_train(model, n_epochs=3 if FAST_MODE else 5):
    criterion = nn.CrossEntropyLoss()
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)
    val_accs = []
    model = model.to(DEVICE)
    for ep in range(n_epochs):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(imgs), lbls)
            loss.backward(); opt.step()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                correct += (model(imgs).argmax(1) == lbls).sum().item()
                total += len(imgs)
        val_accs.append(correct/total)
    return val_accs

results = {}
for mode in ['scratch', 'linear_probe', 'finetune']:
    print(f'Training mode: {mode}')
    model = build_resnet18(mode)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable params: {trainable:,} / {total:,}')
    t0 = time.time()
    accs = quick_train(model)
    elapsed = time.time() - t0
    results[mode] = {'accs': accs, 'time': elapsed}
    print(f'  Best val acc: {max(accs)*100:.1f}%  ({elapsed:.0f}s)')

fig, ax = plt.subplots(figsize=(10, 5))
colors = {'scratch': '#F44336', 'linear_probe': '#FF9800', 'finetune': '#2196F3'}
for mode, result in results.items():
    ep = range(1, len(result['accs']) + 1)
    ax.plot(ep, [a*100 for a in result['accs']], label=mode, color=colors[mode], linewidth=2, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Transfer Learning Strategies: Scratch vs Linear Probe vs Fine-tune\n(ResNet18, EuroSAT)', fontsize=11)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7.2 — Multispectral Input Adaptation

Challenge: ImageNet pretrained models expect 3-channel RGB input.
Sentinel-2 has 13 bands. How do we use pretrained weights?

Strategy 1: Use only RGB bands (B02, B03, B04) — quick and easy
  - Discards 70%+ of spectral information
  - Still often the best approach when spectral info is redundant

Strategy 2: Modify first conv layer
  - Expand first conv from 3 to N input channels
  - Copy pretrained RGB weights to first 3 channels
  - Randomly initialise remaining N-3 channels
  - Fine-tune with small LR for first layer, higher LR for rest

Strategy 3: Project bands to 3-channel input
  - Learn a 1×1 conv (N→3) as a learned spectral combination
  - This is done BEFORE the pretrained backbone
  - Elegant: backbone stays frozen, only the projection is trained

In [ ]:
# Strategy 2: Adapt ResNet for multispectral input
def build_multispectral_resnet(n_bands=10, num_classes=10):
    from torchvision.models import resnet50, ResNet50_Weights
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    
    # Save original first conv weights
    old_conv = model.conv1
    
    # Create new first conv for n_bands input
    model.conv1 = nn.Conv2d(
        n_bands, 64,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )
    
    # Copy pretrained RGB weights to first 3 channels
    with torch.no_grad():
        model.conv1.weight[:, :3] = old_conv.weight  # keep RGB weights
        if n_bands > 3:
            # Initialize remaining channels with near-zero weights
            # (small magnitude → minimal disruption to pretrained features initially)
            nn.init.kaiming_normal_(model.conv1.weight[:, 3:])
            model.conv1.weight[:, 3:] *= 0.1  # scale down initial spectral band weights
    
    # Update classifier
    model.fc = nn.Linear(2048, num_classes)
    
    return model

# Test shape
ms_model = build_multispectral_resnet(n_bands=10, num_classes=10)
x = torch.randn(2, 10, 64, 64)
y = ms_model(x)
print(f'Multispectral input: {x.shape} → logits: {y.shape}')
print(f'First conv weight shape: {ms_model.conv1.weight.shape}')

# Verify the first 3 channels are the pretrained ImageNet weights
from torchvision.models import resnet50, ResNet50_Weights
orig = resnet50(weights=ResNet50_Weights.DEFAULT)
diff = (ms_model.conv1.weight[:, :3] - orig.conv1.weight).abs().max()
print(f'Max diff in RGB channels: {diff:.2e}  (should be ~0, weights preserved)')

## Practical Exercises

### Exercise 7.1 — Layer-wise LR Decay
Train ResNet50 with different learning rates per layer group:
  layer4 + fc: lr = 1e-3
  layer3:      lr = 1e-4
  layer1+2:    lr = 1e-5
Compare to uniform LR. Does layer-wise decay improve accuracy?

### Exercise 7.2 — Domain Shift Analysis
Visualize the feature distributions (using t-SNE) for:
  1. Raw ImageNet features (resnet18, eval mode, no fine-tuning)
  2. Features after 5 epochs of fine-tuning on EuroSAT
Do the EuroSAT classes cluster better after fine-tuning?

### Exercise 7.3 — Multispectral Experiment
Using EuroSAT MS (13-band), compare:
  A) ResNet18 with only RGB bands
  B) ResNet18 with 13-band MS input (Strategy 2)
  C) ResNet18 + 1×1 projection from 13→3 (Strategy 3)
Report accuracy, training time, and which additional bands help most.

### Mini-Project 7
Achieve >95% validation accuracy on EuroSAT RGB using:
  - Progressive fine-tuning (stage 1: head only; stage 2: last block; stage 3: full)
  - Use a learning rate scheduler between stages
  Report the accuracy at each stage and explain the progression.

In [ ]:
print('Chapter 7 Summary:')
print('  Transfer learning from ImageNet works remarkably well in EO')
print('  Early layers: universal features (edges, textures) — always transfer')
print('  Fine-tuning: always better than linear probe if you have > a few hundred samples')
print('  Multispectral: copy RGB weights + initialize extra channels carefully')
print()
print('In Chapter 8: augmentation strategies specific to satellite imagery.')